In [2]:
import json

from utils import parse_orig_sql
from spider_data import get_spider_schema_ddl_and_candidates
from datasets import load_dataset

with open('data/spider/tables.json', 'r', encoding='utf-8') as f:
    spider_tables = json.load(f)

spider_train = load_dataset("xlangai/spider", split="train")

spider_val = load_dataset("xlangai/spider", split="validation")


D:\Master\Grundprojekt\Grundprojekt_Schema_Linking_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from collections import defaultdict
import re

def find_query_multiple_tables(dataset):
    query_multiple_tables = []

    for q in dataset:
        gold_schema = parse_orig_sql(q['query'])
        if len(gold_schema.keys()) > 1:
            query_multiple_tables.append({"query": q['query'], "gold_schema": gold_schema, "db_id": q['db_id']})

    return query_multiple_tables

def tables_connected_by_fk_spider(db_id, table_names):
    """Prueft, ob alle gegebenen Tabellen im FK-Graph der DB zusammenhaengen
    (auch transitiv ueber Zwischentabellen, die nicht im gold_schema stehen)."""
    db = next(db for db in spider_tables if db['db_id'] == db_id)
    orig_tables = db['table_names_original']
    col_names = db['column_names_original']

    name_to_idx = {name.lower(): idx for idx, name in enumerate(orig_tables)}
    target_idxs = [name_to_idx[t.lower()] for t in table_names if t.lower() in name_to_idx]

    if len(target_idxs) < 2:
        return False

    adjacency = {i: set() for i in range(len(orig_tables))}
    for fk_from, fk_to in db['foreign_keys']:
        t_from = col_names[fk_from][0]
        t_to = col_names[fk_to][0]
        adjacency[t_from].add(t_to)
        adjacency[t_to].add(t_from)

    visited = set()
    stack = [target_idxs[0]]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        stack.extend(adjacency[node] - visited)

    return all(idx in visited for idx in target_idxs)

def find_query_tables_fk_connection(dataset):
    query_multiple_tables = find_query_multiple_tables(dataset)
    query_multiple_tables_fk_connected = [
        q for q in query_multiple_tables
        if tables_connected_by_fk_spider(q['db_id'], q['gold_schema'].keys())
    ]

    query_multiple_tables_not_fk_connected = [
        q for q in query_multiple_tables
        if not tables_connected_by_fk_spider(q['db_id'], q['gold_schema'].keys())
    ]

    print(f"verbunden: {len(query_multiple_tables_fk_connected)} / insgesamt: {len(query_multiple_tables)}")
    return query_multiple_tables_fk_connected, query_multiple_tables_not_fk_connected


# --- Spider-Ent: die FKs stehen hier nicht in einer eigenen foreign_keys-Liste,
# sondern als "FOREIGN KEY (...) REFERENCES `table` (...)" direkt im DDL-String
# -> per Regex rausziehen, statt ueber Spalten-Indizes wie bei Spider.
_FK_REF_REGEX = re.compile(r'FOREIGN KEY\s*\([^)]*\)\s*REFERENCES\s*`?([^`\s(]+)`?\s*\(', re.IGNORECASE)

def get_ent_fk_edges(data_asset):
    """Baut die FK-Kanten (Tabelle <-> Tabelle) fuer einen Spider-Ent data_asset aus den DDLs."""
    edges = []
    for table_info in schema_with_parsed_candidates.get(data_asset, []):
        table_name = table_info['candidates']['table']
        if not table_name:
            continue
        for ref_table in _FK_REF_REGEX.findall(table_info['ddl']):
            edges.append((table_name.lower(), ref_table.lower()))
    return edges

def tables_connected_by_fk_ent(data_asset, table_names):
    """Prueft, ob alle gegebenen Tabellen im FK-Graph des Spider-Ent data_assets
    zusammenhaengen (auch transitiv ueber Zwischentabellen, die nicht im gold_schema stehen)."""
    target = {t.lower() for t in table_names if t}

    if len(target) < 2:
        return False

    adjacency = defaultdict(set)
    for t_from, t_to in get_ent_fk_edges(data_asset):
        adjacency[t_from].add(t_to)
        adjacency[t_to].add(t_from)

    start = next(iter(target))
    visited = set()
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        stack.extend(adjacency[node] - visited)

    return all(t in visited for t in target)

def find_query_multiple_tables_ent(dataset):
    query_multiple_tables = []

    for q in dataset:
        gold_schema = get_ent_gold_schema_neu(q)
        if len(gold_schema.keys()) > 1:
            query_multiple_tables.append({
                "query": q['original_SQL'],
                "gold_schema": gold_schema,
                "data_asset": q['data_asset'],
            })

    return query_multiple_tables

def find_query_tables_fk_connection_ent(dataset):
    query_multiple_tables = find_query_multiple_tables_ent(dataset)
    query_multiple_tables_fk_connected = [
        q for q in query_multiple_tables
        if tables_connected_by_fk_ent(q['data_asset'], q['gold_schema'].keys())
    ]

    query_multiple_tables_not_fk_connected = [
        q for q in query_multiple_tables
        if not tables_connected_by_fk_ent(q['data_asset'], q['gold_schema'].keys())
    ]

    print(f"verbunden: {len(query_multiple_tables_fk_connected)} / insgesamt: {len(query_multiple_tables)}")
    return query_multiple_tables_fk_connected, query_multiple_tables_not_fk_connected


In [4]:
print("spider_train")
find_query_tables_fk_connection(spider_train)

print("spider_val")
find_query_tables_fk_connection(spider_val)

spider_train
verbunden: 2940 / insgesamt: 3079
spider_val
verbunden: 433 / insgesamt: 459


([{'query': 'SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id',
   'gold_schema': {'stadium': ['stadium_id', 'name'],
    'concert': ['stadium_id']},
   'db_id': 'concert_singer'},
  {'query': 'SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id',
   'gold_schema': {'stadium': ['stadium_id', 'name'],
    'concert': ['stadium_id']},
   'db_id': 'concert_singer'},
  {'query': 'SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1',
   'gold_schema': {'stadium': ['stadium_id', 'capacity', 'name'],
    'concert': ['stadium_id', 'year']},
   'db_id': 'concert_singer'},
  {'query': 'select t2.name ,  t2.capacity from concert as t1 join stadium as t2 on t1.stadium_id  =  t2.stadium_id where t1.year  >  2013 group by t2.stadiu

In [5]:
from spider_ent_data import get_spider_ent_data, get_ent_gold_schema_neu, schema_with_parsed_candidates
from spider_ent_data import spider_ent as spider_ent_raw

spider_ent = get_spider_ent_data()
print()

In [6]:
print("spider_ent")
find_query_tables_fk_connection_ent(spider_ent_raw)


spider_ent
verbunden: 237 / insgesamt: 241


([{'query': 'SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id',
   'gold_schema': {'zentra11_concer_ven_stadium': ['venue_id', 'venue_name'],
    'zentra11_concer_evt_concert': ['venue_id']},
   'data_asset': 'arts_culture_and_media'},
  {'query': 'SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id',
   'gold_schema': {'zentra11_concer_ven_stadium': ['venue_id', 'venue_name'],
    'zentra11_concer_evt_concert': ['venue_id']},
   'data_asset': 'arts_culture_and_media'},
  {'query': 'SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1',
   'gold_schema': {'zentra11_concer_ven_stadium': ['venue_id',
     'venue_cap_max',
     'venue_name'],
    'zentra11_concer_evt_concert': ['venue_id', 'concert_year']},
   'data_asse